In [5]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv")
df.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

df = df.fillna(0)

y = df.fuel_efficiency_mpg
X = df.drop(columns=["fuel_efficiency_mpg"])

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=1)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=1)

dv = DictVectorizer(sparse=True)
X_train_dict = X_train.to_dict(orient='records')
X_val_dict = X_val.to_dict(orient='records')

X_train_dv = dv.fit_transform(X_train_dict)
X_val_dv = dv.transform(X_val_dict)

In [7]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train_dv, y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,1
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,1
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


Question 1. Most important featute

In [8]:
from sklearn.tree import export_text
print(export_text(dt, feature_names=dv.feature_names_))

|--- vehicle_weight <= 3028.82
|   |--- value: [16.86]
|--- vehicle_weight >  3028.82
|   |--- value: [12.87]



Question 2. RMSE on validation 

In [9]:
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from math import sqrt

rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)
rf.fit(X_train_dv, y_train)

y_pred = rf.predict(X_val_dv)
rmse = sqrt(mean_squared_error(y_val, y_pred))
print("Validation RMSE:", rmse)


Validation RMSE: 0.4602815367032658


Question 3. Number of estimators

In [10]:
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from math import sqrt

rmse_values = []

for n in range(10, 210, 10):
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )
    rf.fit(X_train_dv, y_train)
    y_pred = rf.predict(X_val_dv)
    rmse = sqrt(mean_squared_error(y_val, y_pred))
    rmse_values.append((n, round(rmse, 3)))

for n, rmse in rmse_values:
    print(f"n_estimators={n:<3} -> RMSE: {rmse}")

best_rmse = min([r for _, r in rmse_values])
best_n = [n for n, r in rmse_values if r == best_rmse][0]
print("\nBest n_estimators:", best_n)
print("Best RMSE:", best_rmse)


n_estimators=10  -> RMSE: 0.46
n_estimators=20  -> RMSE: 0.446
n_estimators=30  -> RMSE: 0.44
n_estimators=40  -> RMSE: 0.438
n_estimators=50  -> RMSE: 0.437
n_estimators=60  -> RMSE: 0.436
n_estimators=70  -> RMSE: 0.436
n_estimators=80  -> RMSE: 0.436
n_estimators=90  -> RMSE: 0.435
n_estimators=100 -> RMSE: 0.435
n_estimators=110 -> RMSE: 0.435
n_estimators=120 -> RMSE: 0.435
n_estimators=130 -> RMSE: 0.435
n_estimators=140 -> RMSE: 0.435
n_estimators=150 -> RMSE: 0.435
n_estimators=160 -> RMSE: 0.435
n_estimators=170 -> RMSE: 0.435
n_estimators=180 -> RMSE: 0.435
n_estimators=190 -> RMSE: 0.435
n_estimators=200 -> RMSE: 0.435

Best n_estimators: 90
Best RMSE: 0.435


Question 4. Best max_depth

In [ ]:
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import numpy as np

depth_values = [10, 15, 20, 25]
n_values = range(10, 201, 10)

results = {}

for depth in depth_values:
    rmses = []
    for n in n_values:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )
        rf.fit(X_train_dv, y_train)
        y_pred = rf.predict(X_val_dv)

        rmse = root_mean_squared_error(y_val, y_pred)
        rmses.append(rmse)   

    results[depth] = np.mean(rmses)
    print(f"max_depth={depth}, mean RMSE={np.mean(rmses):.4f}")

best_depth = min(results, key=results.get)
print(f"\n Best max_depth: {best_depth} with mean RMSE={results[best_depth]:.4f}")

max_depth=10, mean RMSE=0.4362
max_depth=15, mean RMSE=0.4378
max_depth=20, mean RMSE=0.4377
max_depth=25, mean RMSE=0.4377

 Best max_depth: 10 with mean RMSE=0.4362


Question 5. Most important feature

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)
rf.fit(X_train_dv, y_train)

feature_importance = rf.feature_importances_

importance_df = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': feature_importance
})

importance_df = importance_df.sort_values(by='importance', ascending=False)
print(importance_df.head(10))


                feature  importance
13       vehicle_weight    0.959878
6            horsepower    0.015933
0          acceleration    0.011442
3   engine_displacement    0.003159
7            model_year    0.003066
8         num_cylinders    0.002323
9             num_doors    0.001576
12           origin=USA    0.000496
10          origin=Asia    0.000431
11        origin=Europe    0.000419


Question 6. XGBoost eta

In [11]:
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error

dtrain = xgb.DMatrix(X_train_dv, label=y_train)
dval = xgb.DMatrix(X_val_dv, label=y_val)

watchlist = [(dtrain, 'train'), (dval, 'val')]

xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

model_03 = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)

y_pred_03 = model_03.predict(dval)
rmse_03 = root_mean_squared_error(y_val, y_pred_03)
print(f"RMSE with eta=0.3: {rmse_03:.4f}")

xgb_params['eta'] = 0.1
model_01 = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)

y_pred_01 = model_01.predict(dval)
rmse_01 = root_mean_squared_error(y_val, y_pred_01)
print(f"RMSE with eta=0.1: {rmse_01:.4f}")

if rmse_01 < rmse_03:
    print("\n eta=0.1 gives the best RMSE")
elif rmse_01 > rmse_03:
    print("\n eta=0.3 gives the best RMSE")
else:
    print("\n Both give equal RMSE")


RMSE with eta=0.3: 0.4434
RMSE with eta=0.1: 0.4167

 eta=0.1 gives the best RMSE
